In [1]:
%load_ext autoreload 
%autoreload 2
%reload_ext autoreload

# 1) At the very top of your script or notebook, before you import tensorflow:
import os
# suppress C++ INFO and WARNING logs (0 = all, 1 = filter INFO, 2 = filter WARNING, 3 = filter ERROR)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
    
import shap
from skimage.segmentation import slic
import tensorflow as tf
# force TF to retain *every* intermediate tensor (including control‐flow outputs)
#Required for running gradientexplainer
tf.compat.v1.experimental.output_all_intermediates(True)

import numpy as np
from tensorflow.keras.models import load_model
from function import losses
from tensorflow.keras.layers import SpatialDropout2D, Dropout, Lambda
import time
import keras_cv
from tqdm import tqdm
import random

IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


Using TensorFlow backend


## Code to investigate the impact of the SEED based on selected number of background samples

In [2]:

# 2) Suppress Python warnings (including deprecation warnings)
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

# 3) Import and quiet TensorFlow’s Python logger
import logging

# this silences most tf.get_logger messages
tf.get_logger().setLevel(logging.ERROR)
# and this silences lower-level logs from the 'tensorflow' namespace
logging.getLogger('tensorflow').setLevel(logging.ERROR)


In [3]:
region_name = "CONUS"
experiment = "EX29"
week_lead = 1
ref_source = "GEFSv12"

checkpoint = f'checkpoints/{region_name}/Wk{week_lead}/Wk{week_lead}_{experiment}_regular_RZSM' if \
ref_source == 'GEFSv12' else f'checkpoints/{region_name}/Wk{week_lead}/Wk{week_lead}_{experiment}_ECMWF_regular_RZSM'
print(checkpoint)

checkpoints/CONUS/Wk1/Wk1_EX29_regular_RZSM


In [4]:
#feature channel names (predictors)
with open(f"channel_list_information/Wk{week_lead}/{experiment}_RZSM_channel_list.txt") as f:
    names = [line.strip() for line in f]
names = [s.split()[-1] for s in names]
print(names)

['RZSM_obs_lag-1', 'RZSM_obs_lag-7', 'RZSM_obs_lag-14', 'pwat_obs_lag-1', 'spfh_obs_lag-1', 'tmax_obs_lag-1', 'diff_temp_obs_lag-1', 'z200_obs_lag-1', 'pwat_ref_lead1', 'spfh_ref_lead1', 'tmax_ref_lead1', 'diff_temp_ref_lead1', 'z200_ref_lead1']


In [5]:
def select_n_channels(week_lead, ref_source):
    # return just the channel count. These are known beforehand because of how the models were setup
    if week_lead == 1 and ref_source == 'GEFSv12':
        return 13
    if week_lead == 2 and ref_source == 'GEFSv12':
        return 14
    if week_lead == 3 and ref_source == 'GEFSv12':
        return 5
    if week_lead == 4 and ref_source == 'GEFSv12':
        return 6
    if week_lead == 5 and ref_source == 'GEFSv12':
        return 7
    raise ValueError(f"Unsupported lead={week_lead}, source={ref_source}")

def return_data_dir(week_lead, ref_source,region_name):
    return (f'Data/model_npy_input_data/{region_name}/Wk{week_lead}_EX_input_data' if ref_source == 'GEFSv12' else  f'Data/{model_npy_input_data}/{region_name}/Wk{week_lead}_ECMWF_EX_input_data')


# 1a) Define a function that maps dropout → identity
def remove_dropout(layer):
    """
    Replace any SpatialDropout2D or Dropout layer with a no-op identity layer.
    All other layers are returned unchanged.
    """
    if isinstance(layer, (SpatialDropout2D, Dropout, keras_cv.layers.SqueezeAndExcite2D)):
        return Lambda(lambda x: x)
    return layer

# Load training and testing data

In [6]:


X_train = np.load(f'{return_data_dir(week_lead, ref_source,region_name)}/{experiment}_RZSM_training_input.npy')
X_test = np.load(f'{return_data_dir(week_lead, ref_source,region_name)}/{experiment}_RZSM_testing_input.npy')

print(X_train[0,:,:,0])

[[0.42408124 0.4734531  0.4485158  ... 0.43075514 0.42779055 0.42665896]
 [0.4341465  0.44612762 0.4472022  ... 0.         0.4281042  0.        ]
 [0.4442203  0.44606608 0.4603065  ... 0.42894194 0.4409061  0.45954335]
 ...
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]
 [0.         0.         0.         ... 0.         0.         0.        ]]


# Load model and channel list information

In [7]:
#Add custom loss function
model = load_model(
    checkpoint,
    custom_objects={"crps2d_tf": losses.crps2d_tf},
    compile=False        # you don’t need to recompile if you’re just doing inference/SHAP
)



# Load model weights and prune layers which are not compatible with Gradient Explainer

In [8]:
# 1) Save the original weights
print("▶ Saving original model weights…")
model.save_weights("/tmp/original_weights.h5")
print("✅ Weights saved to /tmp/original_weights.h5\n")

# 2) Clone & prune the model
def prune_layer(layer):
    # replace Dropout, SpatialDropout2D, and SqueezeAndExcite2D with identity
    if isinstance(layer, (Dropout, SpatialDropout2D, keras_cv.layers.SqueezeAndExcite2D)):
        return Lambda(lambda x: x, name=f"{layer.name}_pruned")
    return layer

print("▶ Cloning and pruning model…")
model_no_do = tf.keras.models.clone_model(
    model,
    clone_function=prune_layer
)
print("✅ Clone+prune complete.\n")

# 3) Load weights by name, skipping mismatches
print("▶ Loading matching weights into pruned model (by_name=True, skip_mismatch=True)…")
model_no_do.load_weights(
    "/tmp/original_weights.h5",
    by_name=True,
    skip_mismatch=True
)
print("✅ Weights loaded.\n")

# 4) Rebuild scalar-output model
print("▶ Building scalar-output model…")
final_head   = model_no_do.output[2]                        # 3rd head
scalar       = tf.reduce_mean(final_head, axis=[1,2,3])     # mean over H, W
scalar_model = tf.keras.Model(inputs=model_no_do.input, outputs=scalar)
print("✅ Scalar model built.")
print("   Inputs:",  scalar_model.input.shape)
print("   Output:",  scalar_model.output.shape)

▶ Saving original model weights…
✅ Weights saved to /tmp/original_weights.h5

▶ Cloning and pruning model…
✅ Clone+prune complete.

▶ Loading matching weights into pruned model (by_name=True, skip_mismatch=True)…
✅ Weights loaded.

▶ Building scalar-output model…
✅ Scalar model built.
   Inputs: (None, 48, 96, 13)
   Output: (None,)


# Gradient Explainer

## Runs a different number of background samples to estimate how these impact the variance and shap values of the outputs

In [9]:


# PARAMETERS
Ks         = [10,25,50,100,250]  # background‐sample sizes to test
SEEDS      = [0, 1, 2, 3]              # different random seeds for robustness
batch_size = 10
C          = select_n_channels(week_lead, ref_source)
N_train    = X_train.shape[0]
N_test     = X_test.shape[0]

# Base directory to hold all runs
base_dir = f"Data/shap_tests/Wk{week_lead}"
os.makedirs(base_dir, exist_ok=True)

for seed in SEEDS:
    # 0) seed everything for reproducibility
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    for K in Ks:
        # 1) Sample K backgrounds
        bg_idx = np.random.choice(N_train, size=K, replace=False)
        background = X_train[bg_idx, : ,: , :C]  # (K, H, W, C)

        # 2) Build the explainer once
        explainer = shap.GradientExplainer(scalar_model, background)

        # 3) Compute SHAP values on the test set in batches
        shap_list = []
        print(f"\n▶ Seed={seed}, running SHAP with K={K} backgrounds …")
        for start in tqdm(range(0, N_test, batch_size), desc=f"Seed={seed} K={K}"):
            end   = min(start + batch_size, N_test)
            batch = X_test[start:end, : ,: , :C]
            sv    = explainer.shap_values(batch, nsamples=background.shape[0], rseed=seed)
            shap_list.append(sv)
        shap_values = np.concatenate(shap_list, axis=0)  # (N_test, H, W, C)

        # 4) Compute per-channel statistics
        flat     = shap_values.reshape(-1, C)          # (N_test*H*W, C)
        mean_abs = np.mean(np.abs(flat), axis=0)       # (C,)
        var_shap = np.var(flat, axis=0)                # (C,)

        # 5) Save results in a unique run directory
        run_name = f"seed{seed:03d}_K{K}_gradient_explainer"
        run_dir  = os.path.join(base_dir, run_name)
        os.makedirs(run_dir, exist_ok=True)

        np.save(os.path.join(run_dir, "bg_idx.npy"),      bg_idx)
        np.save(os.path.join(run_dir, "shap_values.npy"), shap_values)
        np.save(os.path.join(run_dir, "mean_abs.npy"),    mean_abs)
        np.save(os.path.join(run_dir, "var.npy"),         var_shap)

        print(f"✔ Saved seed={seed}, K={K} to '{run_dir}'")



▶ Seed=0, running SHAP with K=10 backgrounds …


Seed=0 K=10: 100%|██████████| 115/115 [01:32<00:00,  1.25it/s]


✔ Saved seed=0, K=10 to 'Data/shap_tests/Wk1/seed000_K10_gradient_explainer'

▶ Seed=0, running SHAP with K=25 backgrounds …


Seed=0 K=25: 100%|██████████| 115/115 [02:36<00:00,  1.36s/it]


✔ Saved seed=0, K=25 to 'Data/shap_tests/Wk1/seed000_K25_gradient_explainer'

▶ Seed=0, running SHAP with K=50 backgrounds …


...

✔ Saved seed=3, K=50 to 'Data/shap_tests/Wk1/seed003_K50_gradient_explainer'

▶ Seed=3, running SHAP with K=100 backgrounds …


Seed=3 K=100: 100%|██████████| 115/115 [08:14<00:00,  4.30s/it]


✔ Saved seed=3, K=100 to 'Data/shap_tests/Wk1/seed003_K100_gradient_explainer'

▶ Seed=3, running SHAP with K=250 backgrounds …


Seed=3 K=250: 100%|██████████| 115/115 [20:05<00:00, 10.48s/it]


✔ Saved seed=3, K=250 to 'Data/shap_tests/Wk1/seed003_K250_gradient_explainer'
